# Glint correction — pure least-squares (no OE prior)

Drop-in replacement for `1.0_glint_correction_no_bathy_optx.ipynb` that removes
the OE prior term entirely.  Minimises only the spectral residual:

$$J(x) = \|\,\sigma_\varepsilon^{-1}\,(y_{\rm obs} - F(x))\|^2$$

This is equivalent to the original `lmfit.minimize` inversion but parallelised
over pixels via `jax.vmap` + `optimistix.LevenbergMarquardt`.

**Why this notebook exists:**  OE spatial artifacts arise when the prior pulls
pixels toward the same prior mean — spectrally flat, spatially uniform.  This
notebook tests whether those artifacts disappear without the prior.

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import scipy.ndimage as ndi
import lmfit
import timeit
import xarray as xr
import rioxarray
import jax
import jax.numpy as jnp
from pyproj import CRS
from xcube.core.store import new_data_store
import configparser

from bio_optics.coupled_models import albert_mobley_3C_jax
from bio_optics.inversion import oe_engine
from bio_optics.inversion import lsq_engine_optx as lsq_engine
from bio_optics.image_processing import dask_engine
from bio_optics.surface import air_water
from bio_optics.surface.reflectance import Rrs_surf
from bio_optics.atmosphere import sky_radiance

jax.config.update('jax_enable_x64', True)

## LSQ inversion engine

`lsq_engine.invert_image` lives in `bio_optics/inversion/lsq_engine.py`.
It is identical to `optimistix_engine` except the prior residual
`r_prior = sa_sqrt_inv * (x − x_a)` is omitted, leaving only the data term.

In [ ]:
# lsq_engine is imported in the imports cell above — nothing to define here.

## Parameters and forward model

Same 3C model as `1.0_glint_correction_no_bathy_optx.ipynb`.
No `sigma_a` is needed — this is pure least-squares.

In [ ]:
params_3C = lmfit.Parameters()
params_3C.add('C_0',   value=2,          min=0,     max=100, vary=True)
params_3C.add('C_1',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_2',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_3',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_4',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_5',   value=0,          min=0,     max=100, vary=False)
params_3C.add('C_Y',   value=0.2,        min=0,     max=4,   vary=True)
params_3C.add('C_X',   value=5,          min=0,     max=100, vary=True)
params_3C.add('C_Mie', value=1,          min=0,     max=100, vary=True)
params_3C.add('f_0',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_1',   value=1,          min=0,     max=1,   vary=False)
params_3C.add('f_2',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_3',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_4',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('f_5',   value=0,          min=0,     max=1,   vary=False)
params_3C.add('B_0',   value=1/np.pi,                        vary=False)
params_3C.add('B_1',   value=1/np.pi,                        vary=False)
params_3C.add('B_2',   value=1/np.pi,                        vary=False)
params_3C.add('B_3',   value=1/np.pi,                        vary=False)
params_3C.add('B_4',   value=1/np.pi,                        vary=False)
params_3C.add('B_5',   value=1/np.pi,                        vary=False)
params_3C.add('bb_phy_spec',         value=0.0010,            vary=False)
params_3C.add('bb_Mie_spec',         value=0.0042,            vary=False)
params_3C.add('bb_X_spec',           value=0.0086,            vary=False)
params_3C.add('a_NAP_spec_lambda_0', value=0.041,             vary=False)
params_3C.add('S',                   value=0.014,             vary=False)
params_3C.add('K',                   value=0,                 vary=False)
params_3C.add('S_NAP',               value=0.011,             vary=False)
params_3C.add('n',                   value=-1,                vary=False)
params_3C.add('lambda_0',            value=440,               vary=False)
params_3C.add('lambda_S',            value=500,               vary=False)
params_3C.add('theta_sun',  value=np.radians(30),             vary=False)
params_3C.add('theta_view', value=np.radians(1e-10),          vary=False)
params_3C.add('n1',    value=1,                               vary=False)
params_3C.add('n2',    value=1.33,                            vary=False)
params_3C.add('kappa_0', value=1.0546,                        vary=False)
params_3C.add('zB',    value=100,  min=0.1, max=1000,         vary=False)
params_3C.add('T_W',   value=18,   min=0,   max=40,           vary=False)
params_3C.add('T_W_0', value=20,                              vary=False)
params_3C.add('g_dd',  value=0.02, min=0,   max=10,           vary=True)
params_3C.add('g_dsr', value=1/np.pi, min=0, max=10,          vary=True)
params_3C.add('g_dsa', value=1/np.pi, min=0, max=10,          vary=True)
params_3C.add('d_r',   value=0.01, min=0,   max=0.1,          vary=True)
params_3C.add('fd_d',  value=1,                               vary=False)
params_3C.add('fd_s',  value=1,                               vary=False)
params_3C.add('offset', value=0,  min=0,    max=0.01,         vary=False)

# No sigma_a — not needed for pure least-squares
log_params_3C = ['C_0', 'C_Y', 'C_X', 'C_Mie', 'g_dd', 'g_dsr', 'g_dsa', 'd_r']

# Scalar noise has no effect on x_hat for LSQ — only rescales chi2.
# Set to 1.0 so chi2 = mean(residuals²), a raw model-fit quality metric.
NOISE      = 1.0
TILE_SIZE  = 4096
MAX_STEPS  = 100

## Glint forward model (2D) — identical to NB01

In [ ]:
def forward_glint_2D(fit_param_ds, parameters, wavelengths, pre_3C):
    n2     = float(parameters['n2'].value)
    rho_L  = air_water.fresnel(parameters['theta_view'].value,
                               n1=parameters['n1'].value, n2=n2)
    Ed_d  = np.array(pre_3C['Ed_d'],  dtype=float)
    Ed_sr = np.array(pre_3C['Ed_sr'], dtype=float)
    Ed_sa = np.array(pre_3C['Ed_sa'], dtype=float)
    Ls_Ed = np.array(pre_3C['Ls_Ed'], dtype=float)
    Ed    = Ed_d + Ed_sr + Ed_sa

    def _get(name):
        if name in fit_param_ds:
            return fit_param_ds[name].values
        return fit_param_ds['x_hat'].sel(param=name).values

    g_dd  = _get('g_dd')[...,  np.newaxis]
    g_dsr = _get('g_dsr')[..., np.newaxis]
    g_dsa = _get('g_dsa')[..., np.newaxis]
    d_r   = _get('d_r')[...,   np.newaxis]

    L_s = sky_radiance.L_s(
        float(parameters['fd_d'].value), g_dd, Ed_d,
        float(parameters['fd_s'].value), g_dsr, Ed_sr,
        g_dsa, Ed_sa,
    )
    R_rs_surface  = Rrs_surf(L_s, Ed, rho_L, d_r)
    R_rs_surface += air_water.fresnel(float(parameters['theta_view'].value), n2=n2) * Ls_Ed
    return R_rs_surface.transpose(2, 0, 1)

## Data store

In [ ]:
config = configparser.ConfigParser()
config.read('../../config.ini')
credentials = {k: v.strip() for k, v in config['Credentials'].items()}

store = new_data_store(
    's3', max_depth=5, root='coastal-cubes/sek/',
    storage_options=dict(
        anon=False,
        key=credentials['s3_client_id'],
        secret=credentials['s3_client_secret'],
    )
)

INPUT_PREFIX  = 'helsinki/L2A_land/'
OUTPUT_PREFIX = 'helsinki/temp/'

SCENE_IDS = [
    'ENMAP01-____L2A-DT0000158841_20251019T101424Z_002_V010505_20260206T113145Z',
]

## Run LSQ glint correction

In [ ]:
for scene_id in SCENE_IDS:
    print(f'Processing {scene_id}')

    img         = store.open_data(f'{INPUT_PREFIX}{scene_id}.zarr')
    scene_crs   = img.rio.crs or CRS.from_wkt(img.spatial_ref.attrs['crs_wkt'])
    wavelengths = img.wavelength.values[:80]

    refl  = img['reflectance'].isel(band=slice(0, 80))
    rrs   = (refl.where(refl > -32768) / 10_000) / np.pi
    cloud = (img['cloud'] == 1) | (img['cirrus'] == 1) | (img['haze'] == 1)
    rrs   = rrs.where(~cloud)

    MIN_WATER_PIXELS = 100
    _wc = store.open_data(f'{OUTPUT_PREFIX}{scene_id}-worldcover.zarr')
    _wf = _wc['water_fraction'].values
    _labeled, _ = ndi.label(_wf >= 0.5)
    _sizes = np.bincount(_labeled.ravel())
    _sizes[0] = 0
    ocean_mask = xr.DataArray(
        np.isin(_labeled, np.where(_sizes >= MIN_WATER_PIXELS)[0]),
        coords=_wc['water_fraction'].coords, dims=_wc['water_fraction'].dims,
    )
    rrs = rrs.where(ocean_mask)

    pre_3C   = albert_mobley_3C_jax.precompute(
        wavelengths, fresh=False,
        theta_sun=float(params_3C['theta_sun'].value),
        P=1013.25, AM=1, RH=60, H_oz=0.38, WV=2.5, alpha=1.317, beta=0.2606,
    )
    f_vec_3C = albert_mobley_3C_jax.make_forward_vec(list(params_3C.keys()), pre_3C)

    # build_inversion is reused only for its forward-model projection;
    # sigma_a values are irrelevant — lsq_engine ignores them entirely.
    dummy_sigma_a = {n: 1.0 for n in params_3C if params_3C[n].vary}
    setup_3C = oe_engine.build_inversion(
        params_3C, f_vec_3C, dummy_sigma_a, log_params=log_params_3C
    )

    Rrs_arr = rrs.transpose('y', 'x', 'band').values

    start_t = timeit.default_timer()
    results_lsq = dask_engine.invert_image(
        Rrs_arr, setup_3C, noise=NOISE,
        invert_fn=lsq_engine.invert_image,
        tile_size=TILE_SIZE,
        max_steps=MAX_STEPS,
        use_lm=True,
    )
    print(f'LSQ inversion done in {timeit.default_timer() - start_t:.1f}s')

    # --- save glint params and corrected Rrs ---
    fit_names = results_lsq['fit_names']
    coords_2d = {'y': rrs.y, 'x': rrs.x}
    glint_ds = xr.Dataset({
        'x_hat': xr.DataArray(
            results_lsq['x_hat'],
            dims=('y', 'x', 'param'),
            coords={**coords_2d, 'param': fit_names},
        ),
        'chi2': xr.DataArray(
            results_lsq['chi2'],
            dims=('y', 'x'),
            coords=coords_2d,
        ),
        'n_steps': xr.DataArray(
            results_lsq['n_steps'],
            dims=('y', 'x'),
            coords=coords_2d,
        ),
    }).rio.write_crs(scene_crs)
    store.write_data(glint_ds,
                     f'{OUTPUT_PREFIX}{scene_id}-glint_params_lsq.zarr', replace=True)

    glint_arr = forward_glint_2D(glint_ds, params_3C, wavelengths, pre_3C)
    glint_da  = xr.DataArray(glint_arr, coords=rrs.coords, dims=rrs.dims).rio.write_crs(scene_crs)
    Rrs_corr  = (rrs - glint_da).rio.write_crs(scene_crs)
    store.write_data(Rrs_corr.to_dataset(name='Rrs'),
                     f'{OUTPUT_PREFIX}{scene_id}-Rrs_lsq.zarr', replace=True)
    print(f'Saved corrected Rrs.')

## Diagnostics — fit quality

`chi2 = mean(residuals²)` — raw model-fit quality.  Scalar noise does not affect
`x_hat` for LSQ, so there is no meaningful chi2 target to calibrate against.
Use chi2 as a spatial quality flag: high values indicate poor model fit.

In [ ]:
import hvplot
import holoviews as hv
import hvplot.xarray

_scene = SCENE_IDS[0]
_gp    = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_lsq.zarr')
_chi2  = _gp['chi2'].values
_finite = _chi2[np.isfinite(_chi2)]

print(f'chi2 = mean(residuals²)  [NOISE=1.0, no normalisation]')
print(f'  median : {np.median(_finite):.4g}')
print(f'  p5–p95 : {np.percentile(_finite, 5):.4g} – {np.percentile(_finite, 95):.4g}')
print(f'  max    : {np.max(_finite):.4g}')

## Maps

In [ ]:
_scene = SCENE_IDS[0]
_wl    = store.open_data(f'{INPUT_PREFIX}{_scene}.zarr').wavelength.values[:80]

_img   = store.open_data(f'{INPUT_PREFIX}{_scene}.zarr')
_refl  = _img['reflectance'].isel(band=slice(0, 80))
_rrs   = (_refl.where(_refl > -32768) / 10_000) / np.pi
_cloud = (_img['cloud'] == 1) | (_img['cirrus'] == 1) | (_img['haze'] == 1)
_wc    = store.open_data(f'{OUTPUT_PREFIX}{_scene}-worldcover.zarr')
_wf    = _wc['water_fraction'].values
_lab, _ = ndi.label(_wf >= 0.5)
_sz    = np.bincount(_lab.ravel()); _sz[0] = 0
_omask = xr.DataArray(np.isin(_lab, np.where(_sz >= 100)[0]),
                      coords=_wc['water_fraction'].coords, dims=_wc['water_fraction'].dims)
_rrs   = _rrs.where(~_cloud).where(_omask)

_Rrs_lsq = store.open_data(f'{OUTPUT_PREFIX}{_scene}-Rrs_lsq.zarr')['Rrs']
_Rrs_lsq = _Rrs_lsq.where(np.isfinite(_Rrs_lsq).any('band'))

# Erode mask to remove mixed land-water edge pixels for display only
from scipy.ndimage import binary_erosion
_eroded = binary_erosion(_omask.values, iterations=3)
_Rrs_lsq_plot = _Rrs_lsq.where(_eroded)

r_idx = int(np.argmin(np.abs(_wl - 670)))
g_idx = int(np.argmin(np.abs(_wl - 550)))
b_idx = int(np.argmin(np.abs(_wl - 460)))
_opts = dict(x='x', y='y', bands='band', framewise=True, aspect='equal',
             width=500, height=500, robust=True)

panel_orig = _rrs.isel(band=[r_idx,g_idx,b_idx]).hvplot.rgb(title='Original Rrs', **_opts)
panel_lsq  = _Rrs_lsq_plot.isel(band=[r_idx,g_idx,b_idx]).hvplot.rgb(title='Glint-corrected Rrs (LSQ)', **_opts)
(panel_orig + panel_lsq)

In [ ]:
_gp   = store.open_data(f'{OUTPUT_PREFIX}{_scene}-glint_params_lsq.zarr')
_valid = np.isfinite(_gp['x_hat']).any('param')
_x_hat = _gp['x_hat'].where(_valid)
_chi2  = _gp['chi2'].where(_valid)
_nsteps = _gp['n_steps'].where(_valid) if 'n_steps' in _gp else None
_opts2 = dict(x='x', y='y', aspect='equal', width=300, height=300, robust=True)

p_chi2 = _chi2.hvplot(cmap='RdYlGn_r', title='chi2 = mean(residuals²)', **_opts2)

if _nsteps is not None:
    _ns = _nsteps.values
    _ns_valid = _ns[_ns >= 0]
    print(f"mean steps: {_ns_valid.mean():.1f}  |  % at max_steps: {100*(_ns_valid == int(_ns_valid.max())).mean():.1f}%")
    p_nsteps = _nsteps.hvplot(cmap='YlOrRd', title='n_steps', **_opts2)
else:
    p_nsteps = None
    print("n_steps not in zarr — re-run invert_image_lsq with updated engine to get step counts")

p_gdd  = _x_hat.sel(param='g_dd').hvplot( cmap='viridis', title='g_dd',  **_opts2)
p_gdsr = _x_hat.sel(param='g_dsr').hvplot(cmap='viridis', title='g_dsr', **_opts2)
p_gdsa = _x_hat.sel(param='g_dsa').hvplot(cmap='viridis', title='g_dsa', **_opts2)
p_dr   = _x_hat.sel(param='d_r').hvplot(  cmap='viridis', title='d_r',   **_opts2)

panels = [p_chi2] + ([p_nsteps] if p_nsteps is not None else []) + [p_gdd, p_gdsr, p_gdsa, p_dr]
hv.Layout(panels).cols(3)